## Imports

In [0]:
#imports
from functools import reduce
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

## Test func

In [0]:
# Test func
def greet():
    print("Hello World")

## Cleaning summarisation functions
These functions are the cumulative steps done in the Cleaning Sandbox file

In [0]:
def transform_pin_data(df):
   
    df = df.dropDuplicates(['unique_id'])

    # Define invalid values for replacement
    invalid_values = {
        "description": ["No description available Story format", "No description available"],
        "follower_count": ["User Info Error"],
        "image_src": ["Image src error."],
        "poster_name": ["User Info Error"],
        "tag_list": ["N,o, ,T,a,g,s, ,A,v,a,i,l,a,b,l,e"],
        "title": ["No Title Data Available"]
    }

    df = df.withColumn("downloaded", F.col("downloaded").cast("boolean"))

    # Convert 'follower_count' to integers (handling 'k' and 'm' suffixes)
    df = df.withColumn(
        "follower_count",
        F.when(F.col("follower_count").rlike(r"^\d+(\.\d+)?[kKmM]?$"),  # Check for numeric, k, or m suffix
               F.regexp_replace(F.regexp_replace(F.lower(F.col("follower_count")), "k", "000"), "m", "000000").cast("int"))
        .otherwise(None)
    )

    # Replace invalid values with None
    for column, values in invalid_values.items():
        df = df.withColumn(column, F.when(F.col(column).isin(values), None).otherwise(F.col(column)))

    # Clean 'save_location' by removing 'Local save in ' prefix
    df = df.withColumn(
        "save_location",
        F.when(F.col("save_location").startswith("Local save in "), 
               F.regexp_replace(F.col("save_location"), r"^Local save in ", ""))
        .otherwise(None)
    )

    # Filter valid 'image_src' values (only keep those matching "https://i.pinimg.com/")
    df = df.withColumn("image_src", F.when(F.col("image_src").rlike(r"^https://i\.pinimg\.com/.*"), F.col("image_src")).otherwise(None))

    df = df.withColumnRenamed("index", "ind")

    column_order = [
        "ind",
        "unique_id",
        "title",
        "description",
        "follower_count",
        "poster_name",
        "tag_list",
        "is_image_or_video",
        "image_src",
        "save_location",
        "category"
    ]

    df = df.select(column_order)
    #df = df.orderBy("ind")  

    return df

In [0]:
def transform_geo_data(df):
    df = df.dropDuplicates(['ind'])

    df = df.withColumn("coordinates", F.array(F.col("latitude"), F.col("longitude")))

    df = df.drop("latitude", "longitude")

    df = df.withColumn("timestamp", F.to_timestamp("timestamp"))

    column_order = [
        "ind", 
        "country", 
        "coordinates", 
        "timestamp"
        ]
    
    df = df.select(column_order)
    #df = df.orderBy("ind")

    return df

In [0]:
def transform_user_data(df):
    df = df.dropDuplicates(['ind'])

    df = df.withColumn("user_name", F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")))
    df = df.drop("first_name", "last_name")

    df = df.withColumn("date_joined", F.to_timestamp("date_joined"))

    column_order = [
        "ind", 
        "user_name", 
        "age", 
        "date_joined"
        ]
    
    df = df.select(column_order)
    #df = df.orderBy("ind")

    return df

## Kinesis Functions

In [0]:
def create_stream_df(stream_name):
    df = spark \
    .readStream \
    .format('kinesis') \
    .option('streamName', stream_name) \
    .option('initialPosition','earliest') \
    .option('region','us-east-1') \
    .option('awsAccessKey', ACCESS_KEY) \
    .option('awsSecretKey', SECRET_KEY) \
    .load()
    return df

In [0]:
def raw_stream_json_to_df(partition_key, schema):
    df = create_stream_df('Kinesis-Prod-Stream')
    df = df.filter(df.partitionKey==partition_key)
    df = df.selectExpr("CAST(data as STRING) jsonData")
    df = df.select(from_json("jsonData", schema).alias("data")).select("data.*")
    return df 

In [0]:
def write_stream_df_to_delta_table(df, stream_name, table_name):
    df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"/tmp/kinesis/_checkpoints/{stream_name}/") \
    .table(table_name)

## Kafka Functions

In [0]:
def get_dataframe_from_drive(topic):
    
    file_location = f"s3a://user-038444ac863e-bucket/topics/{topic}/partition=0/*.json" 
    file_type = "json"
    infer_schema = "true"

    df = spark.read.format(file_type) \
    .option("inferSchema", infer_schema) \
    .load(file_location)

    return df

### Custom Cleaning Functions for Sanity Checks

In [0]:
def get_null_counts_df(df):

    # Here we get the column names and cast them as a Spark column object.
    # The .isNull() will get  aboolean for each value which is then turned into an int 
        # 0 for False & 1 for True
    # The .alias() will rename the column to the original name
    # This is then put to a new df using .select()
    null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
    return null_counts
    

In [0]:
def filter_mostly_numeric_strings(df: DataFrame, threshold: float = 0.5, exclude_cols: list = None):
    
    # Regex pattern for fully numeric values (e.g., "123", "45.67")
    numeric_regex = r'^\d+(\.\d+)?$'  
    
    string_cols = [col_name for col_name, dtype in df.dtypes if dtype == "string"]
    
    if exclude_cols:
        string_cols = [col for col in string_cols if col not in exclude_cols]

    filtered_dfs = {}

    for col_name in string_cols:
        numeric_count = F.length(F.regexp_replace(F.col(col_name), r'\D', ''))  
        
        total_length = F.length(F.col(col_name))  

        # Condition: At least `threshold`% of characters must be numeric
        mostly_numeric_condition = (numeric_count / total_length) >= threshold  
        
        # Condition: Fully numeric values based on regex match
        fully_numeric_condition = F.col(col_name).rlike(numeric_regex)  

        # Apply the combined filter: Keep rows that meet either condition
        filtered_dfs[col_name] = df.filter(mostly_numeric_condition | fully_numeric_condition)

    # Display results for each column
    for col_name, filtered_df in filtered_dfs.items():
        print(f"🔍 Searching for Potential Numeric Issues in Column: {col_name}")
        display(filtered_df)

In [0]:
def filter_invalid_rows(df, invalid_patterns):
    conditions = []
    invalid_columns_expr = F.array()
    invalid_values_expr = F.array()

    # Convert patterns into regex expressions
    regex_patterns = [fr"(?i).*{pattern.replace(' * ', '.*')}.*" for pattern in invalid_patterns]

    for column_name, column_type in df.dtypes:
        if column_type == "string":  
            for regex, pattern in zip(regex_patterns, invalid_patterns):
                condition = F.col(column_name).rlike(regex)  # Match regex pattern
                conditions.append(condition)

                # Capture column names where an invalid substring is found
                invalid_columns_expr = F.array_union(
                    invalid_columns_expr, F.when(condition, F.array(F.lit(column_name))).otherwise(F.array())
                )

                # Capture specific invalid substrings found in a row
                invalid_values_expr = F.array_union(
                    invalid_values_expr, F.when(condition, F.array(F.lit(pattern))).otherwise(F.array())
                )

    # Combine conditions using logical OR (|) for filtering
    filter_condition = reduce(lambda a, b: a | b, conditions)

    # Add two new columns with detected invalid values
    df_filtered = df.withColumn("invalid_columns", invalid_columns_expr).withColumn("invalid_values", invalid_values_expr).filter(filter_condition)

    return df_filtered